In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

In [5]:
load_dotenv()

True

In [6]:
@tool
def add(a: int, b: int) -> int:
    """This functions adds the 2 numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """This function gives the product of 2 numbers"""
    return a * b

In [17]:
llm = ChatOpenAI(model='gpt-5.4-mini')

In [18]:
llm_with_tool = llm.bind_tools([add, multiply])

In [19]:
message_history = []

In [20]:
query = HumanMessage("Give me 5 + 8")
message_history.append(query)

In [21]:
res = llm_with_tool.invoke(message_history)

In [25]:
# Here the llm gives the tools which we need to use
res.tool_calls

[{'name': 'add',
  'args': {'a': 5, 'b': 8},
  'id': 'call_ueoNuCjWtuGbx3IL0NEwMwLq',
  'type': 'tool_call'}]

In [23]:
message_history.append(res)

In [24]:
message_history

[HumanMessage(content='Give me 5 + 8', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 163, 'total_tokens': 183, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EAbqGNa3dxfkKeBZ3DvxA2AQRZK97', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fe1aa-240f-75f0-af79-4def44801b00-0', tool_calls=[{'name': 'add', 'args': {'a': 5, 'b': 8}, 'id': 'call_ueoNuCjWtuGbx3IL0NEwMwLq', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 163, 'output_tokens': 20, 'total_tokens': 183, 'input_token_details': {'a

In [27]:
tool_result = multiply.invoke(res.tool_calls[0]['args'])

In [28]:
tool_result

40

##  Real-World Example: Currency Conversion Tool

In [32]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


convert.args

get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

# tool binding
llm = ChatOpenAI()

llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

messages

ai_message = llm_with_tools.invoke(messages)

messages.append(ai_message)

ai_message.tool_calls

import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



messages

PermissionDeniedError: Error code: 403 - {'error': {'message': 'Project `proj_SEETQITptRSQ42Vsiu9GFpne` does not have access to model `gpt-3.5-turbo`', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}